In [48]:
import pandas as pd

# Load both datasets from local Colab storage
df1 = pd.read_csv('/content/diabetes_prediction_dataset.csv')  # Diabetes dataset
df2 = pd.read_csv('/content/heart_disease_uci.csv')  # Heart disease dataset

# Check column names of diabetes dataset
print(df1.columns.tolist())

# Uncomment to check heart disease columns
# print(df2.columns.tolist())

# Uncomment to check class balance in target columns
# print(df1['diabetes'].value_counts())  # Check diabetic vs non-diabetic count
# print(df2['num'].value_counts())  # Check heart disease severity distribution

# Preview full diabetes dataframe
print(df1)

['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']
       gender   age  hypertension  heart_disease smoking_history    bmi  \
0      Female  80.0             0              1           never  25.19   
1      Female  54.0             0              0         No Info  27.32   
2        Male  28.0             0              0           never  27.32   
3      Female  36.0             0              0         current  23.45   
4        Male  76.0             1              1         current  20.14   
...       ...   ...           ...            ...             ...    ...   
99995  Female  80.0             0              0         No Info  27.32   
99996  Female   2.0             0              0         No Info  17.37   
99997    Male  66.0             0              0          former  27.83   
99998  Female  24.0             0              0           never  35.42   
99999  Female  57.0             0              0

In [49]:
from sklearn.preprocessing import LabelEncoder

# Encode 'gender' column - convert Male/Female to numbers
le1 = LabelEncoder()
df1['gender'] = le1.fit_transform(df1['gender'])

# Encode 'smoking_history' column - convert categories to numbers
le2 = LabelEncoder()
df1['smoking_history'] = le2.fit_transform(df1['smoking_history'])

# Drop blood_glucose_level - direct diagnostic marker, causes data leakage
df1 = df1.drop(columns=['blood_glucose_level'])

# Preview cleaned dataframe
df1

,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,diabetes
0,0,80.0,0,1,4,25.19,6.6,0
1,0,54.0,0,0,0,27.32,6.6,0
2,1,28.0,0,0,4,27.32,5.7,0
3,0,36.0,0,0,1,23.45,5.0,0
4,1,76.0,1,1,1,20.14,4.8,0
...,...,...,...,...,...,...,...,...
99995,0,80.0,0,0,0,27.32,6.2,0
99996,0,2.0,0,0,0,17.37,6.5,0
99997,1,66.0,0,0,3,27.83,5.7,0
99998,0,24.0,0,0,4,35.42,4.0,0


In [50]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define features (X) and target (y)
X = df1.drop(columns='diabetes')  # All columns except target
y = df1['diabetes']  # Target - 0 = not diabetic, 1 = diabetic

# Split into 67% training and 33% testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

# Train Random Forest with class_weight='balanced' to handle imbalanced dataset
# Diabetic patients are 3x less than non-diabetic so balanced weights fix this
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

# Generate predictions on test data
y_pred = rf.predict(X_test)

# Evaluate model performance
acc = accuracy_score(y_pred, y_test)
report = classification_report(y_pred, y_test)
print(report)

              precision    recall  f1-score   support

           0       0.98      0.96      0.97     31029
           1       0.51      0.72      0.59      1971

    accuracy                           0.94     33000
   macro avg       0.74      0.84      0.78     33000
weighted avg       0.95      0.94      0.95     33000



In [139]:
def predict_diabetes(gender, age, hypertension, heart_disease, smoking_history, bmi, HbA1c_level):

    # Validate age range
    if age < 0 or age > 120:
        return "Please enter a valid age between 0 and 120"

    # Validate BMI range
    if bmi < 0 or bmi > 70:
        return "Please enter a valid BMI between 0 and 70"

    # Validate gender against known encoder values
    if gender not in le1.classes_:
        return f"Invalid gender. Please enter one of: {le1.classes_}"

    # Validate smoking history against known encoder values
    if smoking_history not in le2.classes_:
        return f"Invalid smoking history. Please enter one of: {le2.classes_}"

    # Encode text inputs using fitted encoders from training
    gender_encoded = le1.transform([gender])[0]
    smoking_encoded = le2.transform([smoking_history])[0]

    # Create single row dataframe with encoded values for prediction
    data = pd.DataFrame([[gender_encoded, age, hypertension, heart_disease, smoking_encoded, bmi, HbA1c_level]],
                   columns=['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level'])

    # Predict class and probability
    prediction = rf.predict(data)[0]
    probability = rf.predict_proba(data)[0][1]

    if prediction == 1:
        return f"High diabetes risk. Probability: {probability:.0%}"
    else:
        return f"Low diabetes risk. Probability: {probability:.0%}"

In [140]:
# Test prediction - low risk case: young female, healthy BMI, no hypertension, non-smoker
print(predict_diabetes('Female', 90, 0, 0, 'never', 22.0, 5.5))

Low diabetes risk. Probability: 21%


HEART DISEASE MODEL


In [138]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import pandas as pd

df2 = pd.read_csv('heart_disease_uci.csv')

# Drop useless columns
df2 = df2.drop(columns=['id', 'dataset'])

# Drop nulls in key columns
df2 = df2.dropna(subset=['fbs', 'restecg', 'exang', 'slope', 'thal', 'ca', 'trestbps', 'chol', 'thalch', 'oldpeak'])
df2 = df2.reset_index(drop=True)

# Binarize target
df2['num'] = (df2['num'] > 0).astype(int)

# Encode all text columns
le_sex = LabelEncoder()
le_cp = LabelEncoder()
le_fbs = LabelEncoder()
le_restecg = LabelEncoder()
le_exang = LabelEncoder()
le_slope = LabelEncoder()
le_thal = LabelEncoder()

df2['sex'] = le_sex.fit_transform(df2['sex'])
df2['cp'] = le_cp.fit_transform(df2['cp'])
df2['fbs'] = le_fbs.fit_transform(df2['fbs'])
df2['restecg'] = le_restecg.fit_transform(df2['restecg'])
df2['exang'] = le_exang.fit_transform(df2['exang'])
df2['slope'] = le_slope.fit_transform(df2['slope'])
df2['thal'] = le_thal.fit_transform(df2['thal'])

# Define X and y
X1 = df2.drop(columns='num')
y1 = df2['num']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.33, random_state=42)

rf1 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf1.fit(X1_train, y1_train)

y1_pred = rf1.predict(X1_test)
print(classification_report(y1_pred, y1_test))

def predict_heart(age, sex, cp, trestbps, chol, fbs, restecg, thalch, exang, oldpeak, slope, ca, thal):

    if age < 0 or age > 120:
        return "Please enter a valid age between 0 and 120"

    if sex not in le_sex.classes_:
        return f"Invalid sex. Please enter one of: {list(le_sex.classes_)}"

    if cp not in le_cp.classes_:
        return f"Invalid cp. Please enter one of: {list(le_cp.classes_)}"

    if exang not in le_exang.classes_:
        return f"Invalid exang. Please enter one of: {list(le_exang.classes_)}"

    if slope not in le_slope.classes_:
        return f"Invalid slope. Please enter one of: {list(le_slope.classes_)}"

    if thal not in le_thal.classes_:
        return f"Invalid thal. Please enter one of: {list(le_thal.classes_)}"

    # Encode inputs
    sex = le_sex.transform([sex])[0]
    cp = le_cp.transform([cp])[0]
    fbs = le_fbs.transform([fbs])[0]
    restecg = le_restecg.transform([restecg])[0]
    exang = le_exang.transform([exang])[0]
    slope = le_slope.transform([slope])[0]
    thal = le_thal.transform([thal])[0]

    data_2 = pd.DataFrame([[age, sex, cp, trestbps, chol, fbs, restecg, thalch, exang, oldpeak, slope, ca, thal]],
                   columns=['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal'])

    prediction = rf1.predict(data_2)[0]
    probability = rf1.predict_proba(data_2)[0][1]

    if prediction == 1:
        return f"High heart disease risk. Probability: {probability:.0%}"
    else:
        return f"Low heart disease risk. Probability: {probability:.0%}"

# Test
print(predict_heart(63, 'Male', 'asymptomatic', 145, 233, False, 'lv hypertrophy', 150, True, 2.3, 'downsloping', 0, 'fixed defect'))

              precision    recall  f1-score   support

           0       0.89      0.82      0.85        57
           1       0.78      0.86      0.82        42

    accuracy                           0.84        99
   macro avg       0.83      0.84      0.84        99
weighted avg       0.84      0.84      0.84        99

High heart disease risk. Probability: 53%
